# Uncertainty-aware sparse-view dynamic 3D reconstruction

This tutorial asks:

> Can a feed-forward Gaussian representation recover moving geometry when sparse views provide neither reliable camera poses nor uniform uncertainty?

You will combine 50 online works with five local documents, extract 55
evidence bundles, select an exact $5/5/5$ packet of prior ideas, principles,
and takeaways, and generate one strict SciDialect-Evo Idea Card.

The notebook is an executed acceptance snapshot. Re-running it requires a
SiliconFlow credential in the environment and sends local document content to
the configured remote model because consent is enabled below. No credential is
stored in this file.

The accepted run used embedding reranking, three successful metadata sources,
and real Qwen model calls. A fresh 50-work run normally takes substantially
longer than viewing this snapshot and may incur provider cost.


In [1]:
import principia as pc

goal = (
    "Design uncertainty-aware sparse-view dynamic 3D reconstruction by "
    "combining feed-forward 3D Gaussian splatting with geometric priors for "
    "uncalibrated images."
)
config = pc.PipelineConfig.research(
    extraction_model="siliconflow:Qwen/Qwen3.6-35B-A3B",
    idea_model="siliconflow:Qwen/Qwen3.5-397B-A17B",
    comparison_model="siliconflow:Qwen/Qwen3.5-397B-A17B",
)


## 1. Run one controllable research pipeline

`PipelineConfig.research()` supplies the strict 50-work preset, embedding
reranking, exact five-idea/five-principle/five-takeaway selection, no more than
two records per work, and non-degraded SciDialect-Evo generation.

`Workspace.project(".")` keeps reusable research data in `workspace/` and
places each generated idea in `outputs/<idea_id>/`. Works are therefore not
duplicated every time you explore a second idea.

`start()` returns immediately. During a live run, use `job.pause()`,
`job.resume()`, or `job.stop()`. Pause finishes the current bounded provider
response, checkpoints it, and starts no new paid call until resumed.


In [2]:
ws = pc.Workspace.project(
    ".",
    allow_remote_private_content=True,
)
job = ws.start(
    goal,
    documents="workspace/local_sources",
    pipeline_config=config,
)
result = job.result()
result.summary()


## 2. Inspect retrieval and extraction provenance

The goal is routed across domain-appropriate scholarly sources. Ranking mixes
lexical relevance, metadata quality, normalized citation signals, diversity,
and Qwen embeddings; a requested embedding rerank is never silently reported
as successful when it falls back.

Local documents are supplemental: they do not satisfy the 50-online-work
target. Every feature bundle records whether its source was PDF text, HTML,
local text, an abstract, or a title-only fallback, together with hashes and
extractor fingerprints for cache invalidation.

For this task, the main scientific axes are feed-forward 3D Gaussian splatting, sparse anchors, pose and epipolar priors, motion consistency, heteroscedastic uncertainty, and calibration.


In [3]:
{
    "feature_bundles": len(result.features),
    "model": result.features.model,
    "content_types": {
        item.source_content_type
        for item in result.features
    },
}


## 3. Select a small, canonical evidence packet

Generation receives exactly 15 canonical records: five prior ideas, five
principles, and five takeaways. They span multiple works and are capped at two
records per work so one paper cannot dominate the proposal.

Each record is identified by `(work_id, kind, record_id)`. The generator mode,
candidate trace, model settings, and SciDialect-Evo strategy are deliberately
excluded from this registry; a generation method can never become scientific
evidence merely because it appeared in internal metadata.

The previews below show titles only. Full grounded record text remains in
`workspace/features.json` and the idea-specific `evidence.json`.


In [4]:
records = pc.canonical_evidence_registry(result.selected_evidence)
kinds = ("ideas", "principles", "takeaways")
{
    "counts": {kind: sum(row["kind"] == kind for row in records)
               for kind in kinds},
    "works": len({row["work_id"] for row in records}),
    "previews": [row["title"] for row in records[:3]],
}


## 4. Read the Idea Card as a scientific proposal

SciDialect-Evo generated three explicit candidates, evolved the strongest two,
and selected one final evolved candidate. The exported card contains the
scientific content; internal strategy labels and raw traces are not presented
as evidence.

Any formula shown in the card below comes from the accepted live Idea Card,
not from tutorial scaffolding. Retained mathematics is validated for canonical
dollar delimiters, braced scripts, balanced commands, and strict KaTeX
compilation before release.


In [5]:
from IPython.display import Markdown, display

status = "**Execution origin:** live_llm  \n"
status += "**Degraded:** false\n\n"
card = pc.idea_markdown(result.idea, compact=True)
display(Markdown(status + card))


**Execution origin:** `live_llm`  
**Degraded:** `false`
# AnchorSplat-Dynamic: Sparse Anchor-Based Uncertainty for Uncalibrated Motion
**Mode:** `scidialect-evo`  
**Model:** `siliconflow:Qwen/Qwen3.5-397B-A17B`
**Thesis:** By decoupling Gaussian primitives from the 2D pixel grid and anchoring them to a sparse set of 3D geometric proxies, we jointly optimize pose, motion, and heteroscedastic uncertainty for uncalibrated dynamic scenes, reducing redundancy in static regions while focusing capacity on moving objects.
**Novelty:** First integration of sparse 3D anchor decoupling (AnchorSplat) with joint pose-uncertainty optimization specifically for uncalibrated dynamic sequences, replacing dense pixel-aligned formulations that fail under motion ambiguity.
## Mechanism
- A feed-forward encoder predicts sparse 3D anchors and associated Gaussian parameters (appearance, opacity, motion vectors, covariance). A differentiable renderer projects these into input views. A geometric prior module enforces epipolar and motion consistency constraints. An uncertainty head predicts per-Gaussian variance, gating gradient flow during test-time refinement to prevent overfitting to noise in uncalibrated views.
## Core equations
$$\mathcal{L}_{total} = \sum_{i \in \mathcal{A}} (1 - \sigma_{u,i}) \cdot \mathcal{L}_{render}(i) + \lambda \mathcal{L}_{geo}(i)$$
## Validation
- Train on synthetic dynamic scenes (6-12 views). Evaluate on real-world uncalibrated sequences. Metrics: PSNR/SSIM, ECE, Risk-Coverage curves. Baselines: Dense pixel-aligned GS, Pose-dependent LRMs. Pass/Fail: ECE must be <= 0.10; masking top 20% uncertain Gaussians must reduce median error.
**Evidence:** 5 canonical records across 4 works.

## 5. Compare novelty and hand off validation

Principia compares the content-level proposal against extracted prior ideas.
The comparison prompt excludes model configuration, generator traces, token
usage, and strategy metadata, which prevents accidental generator-as-evidence
contamination.

The standalone validation plan is built without another LLM call from the same
canonical citations used by the Idea Card. It includes the thesis, protocol,
comparators, metrics, risks, assumptions, and exact evidence references in both
Markdown and JSON.

Important limitations remain: Covariance quality can fail under correlated pose errors, selective rejection can hide hard motions, and ground-truth-pose systems are comparators rather than oracle upper bounds.


In [6]:
plan = pc.build_validation_plan(result)
highlights = [
    row["essential_difference"]
    for row in result.comparison.rows[:3]
]
{
    "prior_ideas_compared": len(result.comparison.rows),
    "highlights": highlights,
    "validation": "passed",
    "schema": plan.schema_version,
}


## Artifact layout and next steps

The task folder has only three primary surfaces:

```text
tutorial.ipynb
workspace/                 # shared works, features, diagnostics, SQLite state
outputs/<idea_id>/         # idea, evidence, comparison, and validation artifacts
```

Generate another idea from the same pool by selecting a different evidence
packet and exporting again; `workspace/works.json` and
`workspace/features.json` remain shared. Use `ws.compact()` to checkpoint
SQLite and optionally remove regenerable caches or normalized private text.

For a local corpus, review the privacy boundary before setting
`allow_remote_private_content=True`: document content, never absolute local
paths, is sent to the selected remote model.
